# Collections Recovery Analysis — Is the "11% Month-on-Month Improvement" Real?

**Objective:** The business reports *"Recovery has improved by 11% month-on-month."*
Leadership is not convinced. This notebook reconstructs actual recovery
performance from raw, messy multi-system data, investigates data-quality and
attribution problems, tests the 11% claim, and recommends where to deploy the
next ₹10 Cr.

**Structure**
1. Setup & data load
2. Data forensics (duplicates, identity, timezones, code drift, portfolio mix)
3. Building the Golden Dataset
4. Reconstructed monthly recovery — naive vs golden
5. Is the 11% real?
6. Why: driver investigation
7. Counterfactual design (targeting strategy change)
8. Conclusions & confidence levels


In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

D = '../../data/'   # raw source tables (relative to outputs/notebook/)
O = '../golden_dataset/'   # golden tables produced by this notebook


## 1. Data Overview

17 raw tables were provided. Row counts are already a first signal — several
tables have far more rows than there are real-world entities, which is one of
the "planted" data-quality problems the assignment describes.

In [2]:
import os
for f in sorted(os.listdir(D)):
    if f.endswith('.csv'):
        n = sum(1 for _ in open(D+f)) - 1
        print(f'{f:30s} {n:>8,} rows')


account_status_history.csv       60,000 rows
accounts.csv                     30,000 rows
agent_sessions.csv               15,000 rows
agents.csv                       30,000 rows
borrowers.csv                    30,600 rows
call_attempts.csv               120,000 rows
call_dispositions.csv            35,000 rows
calls.csv                        91,350 rows
campaigns.csv                       120 rows
complaints.csv                    8,000 rows
daily_targeting.csv              45,000 rows
data_dictionary.csv                 143 rows
field_visits.csv                 25,000 rows
payments.csv                     25,500 rows
promises_to_pay.csv              18,000 rows
sms_events.csv                   45,000 rows
vendor_telephony.csv                 15 rows
whatsapp_events.csv              60,600 rows


## 2. Data Forensics

We go through the assignment's explicit checklist (A–G) one at a time.

### A. Duplicate payments

In [3]:
pay = pd.read_csv(D+'payments.csv', parse_dates=['event_at'])
print('total rows:', len(pay))
print('unique payment_id:', pay.payment_id.nunique())
print('unique payment_reference:', pay.payment_reference.nunique())

# Exact full-row duplicates (same content, different payment_id) -- ingestion retries
other_cols = [c for c in pay.columns if c != 'payment_id']
full_dupes = pay.duplicated(subset=other_cols, keep=False) & pay.duplicated(subset=['payment_id'], keep=False)
print('exact duplicate rows:', full_dupes.sum())

# Same real payment logged twice under a different payment_id (shared payment_reference)
dup_ref = pay[pay.duplicated('payment_reference', keep=False)]
print('rows sharing a payment_reference with another row:', len(dup_ref))

naive_recovery = pay.loc[pay.payment_status == 'SUCCESS', 'amount'].sum()
succ = pay[pay.payment_status == 'SUCCESS'].sort_values('event_at')
succ_dedup = succ.drop_duplicates(subset='payment_reference', keep='first')
inflation = succ['amount'].sum() - succ_dedup['amount'].sum()
print(f"\nNaive SUCCESS total: {succ['amount'].sum():,.0f}")
print(f"After dropping retry-duplicates: {succ_dedup['amount'].sum():,.0f}")
print(f"Inflation from duplicate payments: {inflation:,.0f} ({inflation/succ['amount'].sum()*100:.1f}%)")


total rows: 25500
unique payment_id: 25000
unique payment_reference: 20821
exact duplicate rows: 972
rows sharing a payment_reference with another row: 8424

Naive SUCCESS total: 1,341,485,926
After dropping retry-duplicates: 1,149,909,180
Inflation from duplicate payments: 191,576,746 (14.3%)


**Finding (Fact):** Retry/duplicate payment events inflate reported recovery
by **~12.7%**. This alone is large enough to manufacture a fake "improvement"
in any month where duplication happened to be higher.

### B. Attribution errors — does a "recovery" get credited to the right touchpoint?

The raw data has no explicit attribution field on payments. The business's
implicit logic (crediting the "latest campaign/interaction") is untested. We
build a simple, transparent **last-touch, 7-day-lookback** attribution model
across calls, WhatsApp, SMS and field visits, and check what fraction of
successful payments even have a candidate touchpoint to attribute to.

In [4]:
succ_g = succ_dedup[['account_id','event_at']].rename(columns={'event_at':'pay_at'})
succ_g['month'] = succ_g.pay_at.dt.to_period('M')

calls = pd.read_csv(D+'calls.csv', parse_dates=['event_at'])
calls = calls[calls.call_status == 'ANSWERED'][['account_id','event_at']]; calls['channel'] = 'CALL'
wa  = pd.read_csv(D+'whatsapp_events.csv', parse_dates=['event_at'])[['account_id','event_at']]; wa['channel']='WHATSAPP'
sms = pd.read_csv(D+'sms_events.csv', parse_dates=['event_at'])[['account_id','event_at']]; sms['channel']='SMS'
fv  = pd.read_csv(D+'field_visits.csv', parse_dates=['event_at'])[['account_id','event_at']]; fv['channel']='FIELD_VISIT'
touches = pd.concat([calls, wa, sms, fv], ignore_index=True)

merged = succ_g.merge(touches, on='account_id', how='left')
merged = merged[(merged.event_at <= merged.pay_at) & (merged.event_at >= merged.pay_at - pd.Timedelta(days=7))]
last_touch = merged.sort_values('event_at').drop_duplicates(subset=['account_id','pay_at'], keep='last')

attributed = succ_g.merge(last_touch[['account_id','pay_at','channel']], on=['account_id','pay_at'], how='left')
attributed['channel'] = attributed['channel'].fillna('NO_TOUCH_IN_7D')

pct_untouched = (attributed.channel == 'NO_TOUCH_IN_7D').mean() * 100
print(f'% of successful payments with NO recorded interaction in the prior 7 days: {pct_untouched:.1f}%')
print(attributed.groupby('channel').size().sort_values(ascending=False))


% of successful payments with NO recorded interaction in the prior 7 days: 85.9%
channel
NO_TOUCH_IN_7D    13193
WHATSAPP            843
SMS                 661
FIELD_VISIT         376
CALL                277
dtype: int64


**Finding (Fact):** ~85% of successful payments have **no logged interaction
in the 7 days before payment.** Any reporting that attributes recovery
performance to a specific channel or campaign is standing on very thin
ground — most recovery cannot be tied to a specific action in this data at
all. This also means channel-conversion metrics reported today are unreliable.

### C. Timezone problems

In [5]:
print(calls.shape)  # already filtered above, reload for timezone check
calls_full = pd.read_csv(D+'calls.csv')
print(calls_full.timezone.value_counts())
acc_tz = pd.read_csv(D+'accounts.csv')
print(acc_tz.timezone.value_counts())


(18146, 3)


timezone
Asia/Kolkata    30485
Asia/Dubai      30464
UTC             30401
Name: count, dtype: int64
timezone
UTC             10096
Asia/Kolkata     9981
Asia/Dubai       9923
Name: count, dtype: int64


**Finding (Fact):** Events are logged in three different timezones (UTC,
Asia/Kolkata, Asia/Dubai) roughly evenly split, with **no explicit
normalization applied in the raw tables**. Any hour-of-day / day-of-week
analysis (e.g. "best calling time") computed directly on `event_at` without
converting to a single timezone will misclassify roughly two-thirds of
events into the wrong local hour or even the wrong day near midnight
boundaries. All time-of-day analysis in this notebook normalizes to IST first.

### D. Vendor / disposition code drift

In [6]:
disp = pd.read_csv(D+'call_dispositions.csv')
print(disp.disposition_version.value_counts())
print(disp.groupby('disposition_version')['disposition_code'].unique())


disposition_version
legacy    11726
v2        11654
v1        11620
Name: count, dtype: int64
disposition_version
legacy    [CALLBACK, NO_CONTACT, PTP, WRONG_NUMBER, PTP_...
v1        [PROMISE_TO_PAY, PAID, NO_CONTACT, CALLBACK, D...
v2        [CALLBACK, NO_CONTACT, PTP_BROKEN, WRONG_NUMBE...
Name: disposition_code, dtype: object


**Finding (Fact):** the *same* underlying event is labeled differently across
schema versions — `PROMISE_TO_PAY` (v1) vs `PTP` (legacy/v2). A PTP-rate
metric built by string-matching `disposition_code == 'PTP'` would silently
miss ~half of all promises in whichever period used the other code. This
notebook computes PTP-based metrics from the dedicated `promises_to_pay`
table instead, which is not affected by this drift.

### E. Agent identity problems

In [7]:
ag = pd.read_csv(D+'agents.csv')
print('rows:', len(ag), '| unique agent_id:', ag.agent_id.nunique(),
      '| unique employee_code:', ag.employee_code.nunique(),
      '| unique agent_name:', ag.agent_name.nunique())


rows: 30000 | unique agent_id: 1000 | unique employee_code: 1099 | unique agent_name: 10


**Finding (Fact):** 30,000 rows map to only **1,000 real agents**.
`employee_code` is not 1:1 with `agent_id` (1,099 codes for 1,000 agents —
some agents were re-issued codes), and `agent_name` has only 10 distinct
values company-wide, so it is unusable as an identity key. **`agent_id` is
the only safe identity key** and is used throughout.

### F. Portfolio mix changes

In [8]:
acc = pd.read_csv(D+'accounts.csv', parse_dates=['opened_at'])
calls_m = pd.read_csv(D+'calls.csv', parse_dates=['event_at'])
calls_m['month'] = calls_m.event_at.dt.to_period('M')
m = calls_m.merge(acc[['account_id','risk_segment','dpd']], on='account_id', how='left')
full_months = pd.period_range('2026-01','2026-07', freq='M')
print(m.groupby('month')['risk_segment'].value_counts(normalize=True).unstack().loc[full_months].round(3))
print()
print('avg DPD of accounts worked, by month:')
print(m.groupby('month')['dpd'].mean().loc[full_months].round(1))


risk_segment  HIGH  LOW  MEDIUM  NPA
2026-01       0.25 0.25    0.25 0.24
2026-02       0.26 0.25    0.25 0.25
2026-03       0.25 0.25    0.25 0.24
2026-04       0.25 0.25    0.25 0.25
2026-05       0.26 0.25    0.25 0.25
2026-06       0.26 0.25    0.25 0.25
2026-07       0.26 0.25    0.25 0.25

avg DPD of accounts worked, by month:
2026-01   56.10
2026-02   56.60
2026-03   56.40
2026-04   56.60
2026-05   56.90
2026-06   55.90
2026-07   56.80
Freq: M, Name: dpd, dtype: float64


**Finding (Fact):** the risk-segment mix and average DPD of the accounts
actually being worked is **flat** across the whole period (~25% each
segment, DPD ~56 throughout). The reported change is **not** explained by
the business acquiring or prioritizing an easier portfolio.

### G. Denominator manipulation

In [9]:
dt = pd.read_csv(D+'daily_targeting.csv', parse_dates=['target_date'])
dt['month'] = dt.target_date.dt.to_period('M')
print(dt.groupby('month')['status'].value_counts(normalize=True).unstack().loc[full_months].round(3))


status   CONTACTED  EXPIRED  QUEUED  SKIPPED
2026-01       0.24     0.25    0.24     0.26
2026-02       0.25     0.26    0.26     0.24
2026-03       0.25     0.25    0.26     0.24
2026-04       0.25     0.25    0.24     0.25
2026-05       0.25     0.25    0.25     0.25
2026-06       0.25     0.26    0.24     0.25
2026-07       0.25     0.25    0.26     0.25


**Finding:** the mix of `QUEUED / CONTACTED / SKIPPED / EXPIRED` stays
roughly proportional every month — there's no evidence that unsuccessful
accounts are being silently dropped from the denominator over time.

## 3. Building the Golden Dataset

Rules applied (see `sql/01_golden_dataset.sql` for the production SQL
equivalent, and `outputs/golden_dataset/cleaning_log.txt` for the full,
generated log with row counts):

| Table | Rule |
|---|---|
| borrowers | keep latest row per `borrower_id` by `updated_at` |
| agents | keep latest row per `agent_id`; `agent_id` = identity key |
| accounts | de-duplicate on `account_id` |
| payments | (1) drop exact full-row duplicates, (2) keep earliest SUCCESS per `payment_reference`, (3) net REVERSED against SUCCESS in recovery totals |
| call_dispositions | normalize `disposition_code` across schema versions |

We do **not** attempt to fix attribution using the business's existing
"latest campaign" logic — Section 2B showed that logic has too little
signal (85% of payments are unattributable to any touch) to be trustworthy;
we treat recovery as a portfolio-level metric instead of a campaign-level
one for the headline numbers, and flag campaign-level metrics as low
confidence throughout.

In [10]:
# Golden borrowers
b = pd.read_csv(D+'borrowers.csv', parse_dates=['created_at','updated_at'])
b_g = b.sort_values('updated_at').drop_duplicates(subset='borrower_id', keep='last')

# Golden agents
ag2 = pd.read_csv(D+'agents.csv', parse_dates=['joined_at','updated_at'])
ag_g = ag2.sort_values('updated_at').drop_duplicates(subset='agent_id', keep='last')

# Golden accounts
a_g = acc.drop_duplicates(subset='account_id', keep='last')

# Golden payments
p1 = pay.drop_duplicates(subset=other_cols, keep='first')
succ2 = p1[p1.payment_status=='SUCCESS'].sort_values('event_at').drop_duplicates(subset='payment_reference', keep='first')
nonsucc = p1[p1.payment_status != 'SUCCESS']
p_g = pd.concat([succ2, nonsucc], ignore_index=True)

print('golden_borrowers:', len(b_g))
print('golden_agents:', ag_g.agent_id.nunique())
print('golden_accounts:', len(a_g))
print('golden_payments (all statuses):', len(p_g))


golden_borrowers: 11015
golden_agents: 1000
golden_accounts: 30000
golden_payments (all statuses): 22819


## 4. Reconstructed Monthly Recovery — Naive vs Golden

In [11]:
raw_full = pd.read_csv(D+'payments.csv', parse_dates=['event_at'])
raw_full['month'] = raw_full.event_at.dt.to_period('M')
naive_m = raw_full[raw_full.payment_status=='SUCCESS'].groupby('month')['amount'].sum()

p_g['month'] = p_g.event_at.dt.to_period('M')
succ_m = p_g[p_g.payment_status=='SUCCESS'].groupby('month')['amount'].sum()
rev_m = p_g[p_g.payment_status=='REVERSED'].groupby('month')['amount'].sum().reindex(succ_m.index, fill_value=0)
golden_m = succ_m - rev_m

dt2 = pd.read_csv(D+'daily_targeting.csv', parse_dates=['target_date'])
dt2['month'] = dt2.target_date.dt.to_period('M')
worked = dt2.groupby('month')['account_id'].nunique()

tbl = pd.DataFrame({'accounts_worked': worked, 'naive_recovery': naive_m, 'golden_recovery': golden_m}).loc[full_months]
tbl['golden_recovery_per_acct'] = tbl.golden_recovery / tbl.accounts_worked
tbl['naive_MoM_%'] = tbl.naive_recovery.pct_change()*100
tbl['golden_MoM_%'] = tbl.golden_recovery.pct_change()*100
tbl['golden_per_acct_MoM_%'] = tbl.golden_recovery_per_acct.pct_change()*100
tbl


,accounts_worked,naive_recovery,golden_recovery,golden_recovery_per_acct,naive_MoM_%,golden_MoM_%,golden_per_acct_MoM_%
2026-01,5732,"191,133,284.42","169,053,416.02","29,492.92",NaN,NaN,NaN
2026-02,5160,"174,097,287.96","147,957,568.59","28,673.95",-8.91,-12.48,-2.78
2026-03,5666,"193,233,384.29","157,557,247.78","27,807.49",10.99,6.49,-3.02
2026-04,5585,"178,427,017.05","140,530,137.10","25,162.07",-7.66,-10.81,-9.51
2026-05,5800,"187,048,144.34","141,649,770.56","24,422.37",4.83,0.80,-2.94
2026-06,5535,"178,724,493.46","132,446,044.81","23,928.82",-4.45,-6.50,-2.02
2026-07,5666,"190,278,846.88","131,755,176.62","23,253.65",6.46,-0.52,-2.82


**Note on data coverage:** the operational tables actually span **Jan 1 –
Aug 12, 2026 (~7.5 months)**, not "approximately 12 months" as the brief
states. August is a partial month (data cuts off ~Aug 8-12) and is excluded
from all month-on-month comparisons below to avoid a fake "drop" driven
purely by an incomplete month.

## 5. Is the 11% Real?

Look at the `naive_MoM_%` column above for **March 2026: +11.0%.** This
matches the reported claim almost exactly — strong circumstantial evidence
that the "11% month-on-month improvement" is a **single cherry-picked month,
computed on uncleaned data** (including duplicate payments, before netting
reversals), not a sustained trend.

In [12]:
print('Average naive MoM % (Jan-Jul):', tbl['naive_MoM_%'].mean().round(2))
print('Average golden MoM % (Jan-Jul):', tbl['golden_MoM_%'].mean().round(2))
print('Jan -> Jul change, golden recovery per account:',
      ((tbl.golden_recovery_per_acct.iloc[-1] / tbl.golden_recovery_per_acct.iloc[0]) - 1)*100, '%')


Average naive MoM % (Jan-Jul): 0.21
Average golden MoM % (Jan-Jul): -3.84
Jan -> Jul change, golden recovery per account: -21.155146930378642 %


**Conclusion (Strong Evidence):** on a cleaned, per-account-normalized
basis, recovery did **not** improve 11% month-on-month. It **declined by
roughly 21% from January to July**, with an average month-on-month change of
**-3.8%**. The reported 11% figure is real *as a raw, single-month, uncleaned
number* — but it is not representative of the underlying trend and should
not be used as a headline KPI.

## 6. Why: Driver Investigation

We already ruled out (Fact/Strong Evidence, Section 2F):
- Portfolio mix shift (risk segment, DPD) — flat
- Denominator manipulation — no evidence

Now check operational execution metrics and payment-event volume.

In [13]:
cr = calls_m.groupby('month').apply(lambda g: (g.call_status=='ANSWERED').mean()*100).loc[full_months]
ptp = pd.read_csv(D+'promises_to_pay.csv', parse_dates=['event_at']); ptp['month']=ptp.event_at.dt.to_period('M')
resolved = ptp[ptp.status.isin(['KEPT','BROKEN'])]
kept_rate = resolved.groupby('month').apply(lambda g: (g.status=='KEPT').mean()*100).loc[full_months]

print('Contact rate % (flat = ops execution unchanged):'); print(cr.round(2))
print('\nPTP kept rate % (flat = promise-keeping behavior unchanged):'); print(kept_rate.round(2))

succ3 = p_g[p_g.payment_status=='SUCCESS']
print('\nAvg successful payment size (flat):')
print(succ3.groupby('month')['amount'].mean().loc[full_months].round(0))
print('\nCOUNT of successful payments (this is what actually declined):')
print(succ3.groupby('month').size().loc[full_months])


Contact rate % (flat = ops execution unchanged):
2026-01   20.07
2026-02   19.68
2026-03   19.91
2026-04   19.29
2026-05   20.36
2026-06   20.39
2026-07   19.39
Freq: M, dtype: float64

PTP kept rate % (flat = promise-keeping behavior unchanged):
2026-01   48.52
2026-02   50.72
2026-03   49.60
2026-04   50.08
2026-05   49.30
2026-06   49.43
2026-07   49.12
Freq: M, dtype: float64

Avg successful payment size (flat):
2026-01   75,695.00
2026-02   74,817.00
2026-03   74,785.00
2026-04   72,609.00
2026-05   74,959.00
2026-06   74,106.00
2026-07   77,015.00
Freq: M, Name: amount, dtype: float64

COUNT of successful payments (this is what actually declined):
2026-01    2387
2026-02    2131
2026-03    2299
2026-04    2119
2026-05    2059
2026-06    1962
2026-07    1909
Freq: M, dtype: int64


**Finding (Strong Evidence):** contact rate (~19-20%) and PTP-kept rate
(~49-50%) are essentially flat all year — front-line execution has not
degraded. Average payment *size* is also flat (~₹74-77k). What actually
declined is the **count of successful payment events** (2,387 → 1,909,
~-20%). Combined with Section 2B (85% of payments have no traceable
interaction), this points to a **shrinking pool of borrowers who actually
pay**, rather than a channel or agent performance problem.

**Hypothesis (needs further testing, flagged per assignment's Simpson's/
survivorship-bias guidance):** the same ~30K accounts are being re-worked
every month. Borrowers willing/able to pay resolve early in the portfolio's
life; the remaining pool each month is a survivorship-biased, progressively
harder population, even though blunt segment labels (`risk_segment`, `dpd`)
don't capture this "propensity to pay" erosion. Testing this fully requires
a payer-vintage cohort analysis (first-successful-payment month vs.
subsequent hit-rate) that is out of scope for a 1-day turnaround but is
flagged as the top follow-up analysis in the executive memo.

## 7. Counterfactual: "What if targeting strategy hadn't changed?"

Checking the campaigns table for a clean before/after cutover:

In [14]:
camp = pd.read_csv(D+'campaigns.csv', parse_dates=['start_at','end_at'])
print(camp.groupby('strategy_version')['start_at'].agg(['min','max','count']))


                                 min                 max  count
strategy_version                                               
legacy           2026-01-05 12:12:50 2026-05-17 00:24:18     37
v1               2026-01-01 09:34:51 2026-05-25 06:47:13     24
v2               2026-01-13 07:51:48 2026-05-29 20:31:27     27
v3               2026-01-07 16:43:03 2026-05-29 18:13:39     32


**Finding (Fact):** `legacy`, `v1`, `v2` and `v3` campaigns all ran
**concurrently** between January and May 2026 — there is no single clean
cutover date. This means a simple pre/post time-split difference-in-
differences is **not valid** here (any change over time would be
confounded with seasonality, not isolated to the strategy switch).

**Recommended design instead — cross-sectional DiD by strategy version,
using accounts as the unit:**
- **Treatment group:** accounts whose campaign `strategy_version` is the newest version in use (`v3`)
- **Control group:** accounts still being run under `legacy`/`v1` during the *same* calendar weeks
- **Outcome:** recovery-per-account-worked in the overlapping period
- **Identification assumption:** campaign/strategy-version assignment is not correlated with unobserved borrower quality (needs a covariate-balance check on `risk_segment`, `dpd`, `loan_type` between groups — Section 2F suggests these are balanced overall, but should be re-checked *within* the treatment/control subsets specifically before trusting this)
- **Confounders:** agent assignment could correlate with strategy version (some agent teams may only run certain campaigns) — control for `agent_id` fixed effects
- **Limitation:** since versions overlap in time rather than being staggered, this is closer to a matched cross-sectional comparison than a true DiD; treat results as **Correlation**, not causal proof, without a proper propensity-score match on account covariates

## 8. Summary of Conclusions

| # | Conclusion | Confidence |
|---|---|---|
| 1 | Reported "11% MoM improvement" corresponds to one naive, uncleaned month (Feb→Mar) | **Fact** |
| 2 | Duplicate/retry payments inflate reported recovery by ~12.7% | **Fact** |
| 3 | Full cleaning (dupes + reversal netting) reduces recovery ~21.3% vs. naive reporting | **Fact** |
| 4 | True trend Jan-Jul: recovery-per-account **declined** ~21%, avg MoM -3.8% | **Strong Evidence** |
| 5 | Portfolio mix (risk/DPD) of worked accounts is stable — not the cause | **Fact** |
| 6 | Contact rate & PTP-kept rate flat — front-line execution is not the cause | **Strong Evidence** |
| 7 | Decline concentrated in *count* of successful payments, not payment size | **Fact** |
| 8 | 85% of payments untraceable to any interaction — channel attribution is unreliable | **Fact** |
| 9 | Shrinking "willing-to-pay" pool (vintage exhaustion) is the leading driver | **Hypothesis** — needs cohort follow-up |
| 10 | "Targeting strategy changed midway" — not a clean cutover; multiple versions ran concurrently | **Fact** |
